# U.S. Recession Probability Model — 12-Month Ahead

A **probit regression** using NBER recession dates as the binary dependent variable,
driven by economic indicators across eight categories.

Based on the methodology of Estrella & Mishkin (1996, 1998), as used by the
NY Fed, Cleveland Fed, and Columbia Threadneedle.

**Framework:** P(Recession\_{t+12} = 1 | X\_t) = Φ(α₀ + α₁X\_t)

Where Φ(·) is the standard normal CDF and X\_t is a vector of economic indicators.

In [ ]:
# Install dependencies
!pip install fredapi statsmodels scikit-learn matplotlib pandas numpy scipy -q

In [ ]:
# ============================================================
# FRED API Key Setup
# Get your free key at: https://fred.stlouisfed.org/docs/api/fred/
# ============================================================
from getpass import getpass

FRED_API_KEY = getpass("Enter your FRED API key: ")

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm
from fredapi import Fred
from scipy import stats
from sklearn.metrics import roc_auc_score, brier_score_loss
import warnings
warnings.filterwarnings("ignore")

fred = Fred(api_key=FRED_API_KEY)
print("FRED connection established.")

## Stage 1: Data Pipeline

Pull 41 FRED series across eight categories. Each category contributes
indicators that capture different dimensions of recession risk.

In [ ]:
# ============================================================
# FRED Series Definitions — 8 Categories
# ============================================================
series_config = {
    # --- National Economic Activity ---
    "CFNAI":    {"name": "Chicago Fed National Activity Index", "category": "National Activity"},
    "CFNAIMA3": {"name": "CFNAI 3-Month Moving Average",      "category": "National Activity"},

    # --- Industrial Indicators ---
    "INDPRO":   {"name": "Industrial Production Index",        "category": "Industrial"},
    "TCU":      {"name": "Capacity Utilization",               "category": "Industrial"},
    "NAPM":     {"name": "ISM Manufacturing PMI",              "category": "Industrial"},

    # --- Consumer Measures ---
    "UMCSENT":  {"name": "U. Michigan Consumer Sentiment",     "category": "Consumer"},
    "DSPIC96":  {"name": "Real Disposable Personal Income",    "category": "Consumer"},

    # --- Labor Market ---
    "UNRATE":   {"name": "Unemployment Rate",                  "category": "Labor"},
    "ICSA":     {"name": "Initial Unemployment Claims",        "category": "Labor"},
    "PAYEMS":   {"name": "Total Nonfarm Payrolls",             "category": "Labor"},

    # --- Inflation ---
    "CPIAUCSL": {"name": "CPI All Urban Consumers",            "category": "Inflation"},
    "PCEPILFE": {"name": "Core PCE Price Index",               "category": "Inflation"},

    # --- Housing ---
    "HOUST":    {"name": "Housing Starts",                     "category": "Housing"},
    "PERMIT":   {"name": "Building Permits",                   "category": "Housing"},

    # --- Banking / Credit ---
    "BAA10YM":  {"name": "Baa Corp Bond - 10Y Treasury Spread", "category": "Banking"},
    "BUSLOANS": {"name": "Commercial & Industrial Loans",      "category": "Banking"},

    # --- Government Bond Yields ---
    "GS10":     {"name": "10-Year Treasury Yield",             "category": "Yields"},
    "TB3MS":    {"name": "3-Month Treasury Bill Rate",         "category": "Yields"},
    "FEDFUNDS": {"name": "Federal Funds Rate",                 "category": "Yields"},

    # --- Target Variable ---
    "USREC":    {"name": "NBER Recession Indicator",           "category": "Target"},
}

print(f"Fetching {len(series_config)} series from FRED...")

In [ ]:
# ============================================================
# Fetch all series from FRED
# ============================================================
OBS_START = "1967-01-01"  # CFNAI starts 1967

raw_data = pd.DataFrame()
failed = []

for sid, info in series_config.items():
    try:
        s = fred.get_series(sid, observation_start=OBS_START)
        # Resample weekly series (ICSA) to monthly
        if sid == "ICSA":
            s = s.resample("MS").mean()
        raw_data[sid] = s
        print(f"  ✓ {sid:12s} — {info['name']}")
    except Exception as e:
        failed.append(sid)
        print(f"  ✗ {sid:12s} — FAILED: {e}")

# Align everything to monthly frequency (month-start)
raw_data.index = pd.to_datetime(raw_data.index)
raw_data = raw_data.resample("MS").last()

print(f"
Fetched {len(series_config) - len(failed)}/{len(series_config)} series.")
print(f"Date range: {raw_data.index.min().strftime('%Y-%m')} to {raw_data.index.max().strftime('%Y-%m')}")
if failed:
    print(f"Failed series: {failed}")
raw_data.tail(3)

## Stage 2: Feature Engineering

Transform raw series into model-ready features:
- **Yield curve spread** (GS10 - TB3MS) — the single most powerful predictor
- **Year-over-year % change** for level series (INDPRO, CPI, Housing Starts, etc.)
- **Levels** for bounded/stationary series (UNRATE, CFNAI, BAA10YM, FEDFUNDS)
- **Dependent variable**: recession indicator shifted 12 months forward

In [ ]:
# ============================================================
# Feature Engineering
# ============================================================
data = raw_data.copy()

# --- Yield curve spread (construct from components for full history) ---
data["SPREAD"] = data["GS10"] - data["TB3MS"]

# --- Year-over-year percent changes for level series ---
yoy_series = ["INDPRO", "CPIAUCSL", "HOUST", "PERMIT", "DSPIC96",
              "PAYEMS", "BUSLOANS", "PCEPILFE"]
for sid in yoy_series:
    if sid in data.columns:
        data[f"{sid}_YOY"] = data[sid].pct_change(12) * 100

# --- ICSA: year-over-year change (rising claims = bad) ---
if "ICSA" in data.columns:
    data["ICSA_YOY"] = data["ICSA"].pct_change(12) * 100

# --- Unemployment rate: 3-month moving average change (Sahm-style) ---
data["UNRATE_CHG3"] = data["UNRATE"].rolling(3).mean() - data["UNRATE"].rolling(3).mean().shift(12)

# --- Dependent variable: recession at month t+12 ---
data["REC_12M"] = data["USREC"].shift(-12)

# --- Alternative: any recession in next 12 months ---
data["REC_ANY_12M"] = data["USREC"].rolling(window=12).max().shift(-12)

print("Features created.")
print(f"Total columns: {len(data.columns)}")
data[["SPREAD", "FEDFUNDS", "BAA10YM", "CFNAI", "INDPRO_YOY",
      "UNRATE", "CPIAUCSL_YOY", "HOUST_YOY", "UMCSENT", "REC_12M"]].tail(5)

## Exploratory Data Analysis

Visualize key indicators against NBER recession periods to validate the data.

In [ ]:
# ============================================================
# Quick EDA: Key indicators vs. recessions
# ============================================================
fig, axes = plt.subplots(4, 2, figsize=(16, 14), sharex=True)
fig.suptitle("Key Recession Indicators vs. NBER Recessions", fontsize=14, y=1.01)

plot_series = [
    ("SPREAD",       "Yield Curve Spread (10Y-3M)",  "bps"),
    ("FEDFUNDS",     "Federal Funds Rate",            "%"),
    ("BAA10YM",      "Baa-10Y Credit Spread",         "ppt"),
    ("CFNAI",        "Chicago Fed National Activity",  "index"),
    ("INDPRO_YOY",   "Industrial Production YoY",      "%"),
    ("UNRATE",       "Unemployment Rate",              "%"),
    ("HOUST_YOY",    "Housing Starts YoY",             "%"),
    ("UMCSENT",      "Consumer Sentiment",             "index"),
]

usrec = data["USREC"].dropna()

for ax, (col, title, unit) in zip(axes.flat, plot_series):
    if col in data.columns:
        s = data[col].dropna()
        ax.plot(s.index, s.values, linewidth=0.9, color="#1f77b4")
    ax.fill_between(usrec.index, ax.get_ylim()[0], ax.get_ylim()[1],
                    where=usrec.values == 1, color="gray", alpha=0.2)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel(unit, fontsize=8)
    ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.show()

## Stage 3: Probit Model Estimation

Three models of increasing complexity:
1. **NY Fed baseline**: Yield curve spread only (Estrella & Mishkin)
2. **Wright extension**: Spread + Federal Funds Rate level
3. **Full multi-category**: One indicator per category (8 categories)

All estimated via **expanding-window pseudo out-of-sample** for honest evaluation.

In [ ]:
# ============================================================
# Prepare model dataset
# ============================================================

# Core features — one per category, chosen for predictive power
FEATURES_FULL = [
    "SPREAD",        # Yields: 10Y-3M spread (most powerful single predictor)
    "FEDFUNDS",      # Yields: Fed funds rate level (Wright 2006)
    "BAA10YM",       # Banking: Baa credit spread
    "CFNAI",         # National activity: Chicago Fed index
    "INDPRO_YOY",    # Industrial: production YoY change
    "UMCSENT",       # Consumer: sentiment
    "UNRATE",        # Labor: unemployment rate
    "CPIAUCSL_YOY",  # Inflation: CPI YoY
    "HOUST_YOY",     # Housing: starts YoY change
]

TARGET = "REC_12M"

# Build clean model dataframe
model_cols = FEATURES_FULL + [TARGET, "USREC"]
model_df = data[[c for c in model_cols if c in data.columns]].dropna()

print(f"Model dataset: {len(model_df)} observations")
print(f"Date range: {model_df.index.min().strftime('%Y-%m')} to {model_df.index.max().strftime('%Y-%m')}")
print(f"Recession months: {int(model_df[TARGET].sum())} ({model_df[TARGET].mean()*100:.1f}%)")
print(f"Features: {[c for c in FEATURES_FULL if c in model_df.columns]}")

In [ ]:
# ============================================================
# Model 1: NY Fed Baseline — Spread only (Estrella & Mishkin)
# Model 2: Wright Extension — Spread + Fed Funds Rate
# Model 3: Full Multi-Category — 9 indicators
# ============================================================

models = {
    "NY Fed (Spread Only)": ["SPREAD"],
    "Wright (Spread + FF)": ["SPREAD", "FEDFUNDS"],
    "Full Multi-Category":  FEATURES_FULL,
}

results = {}

for name, features in models.items():
    avail = [f for f in features if f in model_df.columns]
    y = model_df[TARGET].astype(float)
    X = sm.add_constant(model_df[avail].astype(float))

    try:
        res = sm.Probit(y, X).fit(disp=False, method="bfgs", maxiter=500)
        fitted = res.predict(X)
        results[name] = {
            "model": res,
            "features": avail,
            "fitted": fitted,
            "pseudo_r2": res.prsquared,
        }
        print(f"
{'='*60}")
        print(f"  {name}")
        print(f"{'='*60}")
        print(f"  Pseudo R²:  {res.prsquared:.4f}")
        print(f"  Log-Lik:    {res.llf:.1f}")
        print(f"  AIC:        {res.aic:.1f}")
        print(f"  BIC:        {res.bic:.1f}")
    except Exception as e:
        print(f"  {name}: FAILED — {e}")

print("
" + "="*60)
print("  Model comparison complete.")

In [ ]:
# ============================================================
# Full model coefficient table
# ============================================================
if "Full Multi-Category" in results:
    print(results["Full Multi-Category"]["model"].summary())

## Stage 3b: Expanding-Window Out-of-Sample Estimation

For honest evaluation, each month's probability is generated from a model
trained **only on data available at that point** (expanding window).
Minimum training window: 120 months (10 years).

In [ ]:
# ============================================================
# Expanding-window pseudo out-of-sample probabilities
# ============================================================
MIN_WINDOW = 120  # 10 years minimum training data

# Run OOS for the full multi-category model
oos_features = [f for f in FEATURES_FULL if f in model_df.columns]
oos_probs = pd.Series(index=model_df.index, dtype=float)

n = len(model_df)
print(f"Running expanding-window OOS estimation ({n - MIN_WINDOW} iterations)...")

for i in range(MIN_WINDOW, n):
    train = model_df.iloc[:i]
    y_t = train[TARGET].astype(float)
    X_t = sm.add_constant(train[oos_features].astype(float))

    try:
        res = sm.Probit(y_t, X_t).fit(disp=False, method="bfgs", maxiter=300)
        X_curr = sm.add_constant(
            model_df[oos_features].iloc[[i]].astype(float)
        )
        oos_probs.iloc[i] = res.predict(X_curr)[0]
    except Exception:
        oos_probs.iloc[i] = float("nan")

    if (i - MIN_WINDOW) % 100 == 0:
        pct = (i - MIN_WINDOW) / (n - MIN_WINDOW) * 100
        print(f"  {pct:5.1f}% complete (month {model_df.index[i].strftime('%Y-%m')})")

print("Out-of-sample estimation complete.")
print(f"Valid OOS probabilities: {oos_probs.notna().sum()}")

## Model Evaluation

Compare models using AUROC and Brier Score on in-sample fitted probabilities
and out-of-sample expanding-window probabilities.

In [ ]:
# ============================================================
# Evaluation metrics
# ============================================================
print(f"{"Model":<28s} {"AUROC":<10s} {"Brier":<10s} {"Pseudo R²":<10s}")
print("-" * 58)

y_true = model_df[TARGET].astype(float)

for name, res_dict in results.items():
    fitted = res_dict["fitted"]
    auroc = roc_auc_score(y_true, fitted)
    brier = brier_score_loss(y_true, fitted)
    pr2 = res_dict["pseudo_r2"]
    print(f"{name:<28s} {auroc:<10.4f} {brier:<10.4f} {pr2:<10.4f}")

# OOS metrics
oos_valid = oos_probs.dropna()
if len(oos_valid) > 0:
    y_oos = model_df.loc[oos_valid.index, TARGET].astype(float)
    auroc_oos = roc_auc_score(y_oos, oos_valid)
    brier_oos = brier_score_loss(y_oos, oos_valid)
    print(f"{"Full (Out-of-Sample)":<28s} {auroc_oos:<10.4f} {brier_oos:<10.4f} {"—":<10s}")

## Stage 4: Visualization — Columbia Threadneedle Style

Main chart: 12-month-ahead recession probability with NBER recession shading.
Shows both in-sample fitted (full model) and out-of-sample probabilities.

In [ ]:
# ============================================================
# Main Recession Probability Chart
# ============================================================
fig, ax = plt.subplots(figsize=(16, 6))

# NBER recession shading
usrec = data["USREC"].dropna()
ax.fill_between(usrec.index, 0, 100, where=usrec.values == 1,
                color="#d4d4d4", alpha=0.6, label="NBER Recession")

# In-sample fitted probability (Full model)
if "Full Multi-Category" in results:
    fitted = results["Full Multi-Category"]["fitted"]
    ax.plot(fitted.index, fitted * 100,
            color="#1f77b4", linewidth=1.2, alpha=0.5,
            label="In-Sample (Full Model)")

# Out-of-sample probability
oos_valid = oos_probs.dropna()
if len(oos_valid) > 0:
    ax.plot(oos_valid.index, oos_valid * 100,
            color="#d62728", linewidth=1.5,
            label="Out-of-Sample (Expanding Window)")

# Reference lines
ax.axhline(y=50, color="red", linestyle="--", alpha=0.3, linewidth=0.8,
           label="50% Threshold")
ax.axhline(y=30, color="orange", linestyle=":", alpha=0.3, linewidth=0.8,
           label="30% Warning Level")

# Styling
ax.set_ylim(0, 100)
ax.set_xlim(model_df.index.min(), data.index.max())
ax.set_ylabel("Probability (%)", fontsize=12)
ax.set_title("U.S. Recession Probability — 12-Month Ahead (Probit Model)",
             fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig("recession_probability_main.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to recession_probability_main.png")

## Model Comparison Chart

Compare all three models: NY Fed baseline, Wright extension, and full multi-category.

In [ ]:
# ============================================================
# Model Comparison: All three probit specifications
# ============================================================
colors = {"NY Fed (Spread Only)": "#2ca02c",
          "Wright (Spread + FF)": "#ff7f0e",
          "Full Multi-Category": "#1f77b4"}

fig, ax = plt.subplots(figsize=(16, 6))

# NBER recession shading
ax.fill_between(usrec.index, 0, 100, where=usrec.values == 1,
                color="#d4d4d4", alpha=0.6, label="NBER Recession")

for name, res_dict in results.items():
    fitted = res_dict["fitted"]
    ax.plot(fitted.index, fitted * 100, color=colors[name],
            linewidth=1.0, alpha=0.8, label=f"{name} (R²={res_dict['pseudo_r2']:.3f})")

ax.axhline(y=50, color="red", linestyle="--", alpha=0.3, linewidth=0.8)
ax.set_ylim(0, 100)
ax.set_ylabel("Probability (%)", fontsize=12)
ax.set_title("Model Comparison — 12-Month-Ahead Recession Probability",
             fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig("recession_probability_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Current Recession Probability Reading

The model's latest estimate using the most recent available data.

In [ ]:
# ============================================================
# Current reading: latest recession probability
# ============================================================
if "Full Multi-Category" in results:
    full_res = results["Full Multi-Category"]
    latest_prob = full_res["fitted"].iloc[-1] * 100
    latest_date = full_res["fitted"].index[-1]

    print("=" * 60)
    print("  CURRENT 12-MONTH-AHEAD RECESSION PROBABILITY")
    print("=" * 60)
    print(f"  As of:        {latest_date.strftime('%B %Y')}")
    print(f"  Probability:  {latest_prob:.1f}%")
    print()

    if latest_prob > 50:
        print("  ⚠️  ELEVATED: Probability exceeds 50% — recession more")
        print("     likely than not within 12 months.")
    elif latest_prob > 30:
        print("  ⚠️  WARNING: Probability above 30% warrants attention.")
    else:
        print("  ✅  LOW: Probability below 30% — recession risk contained.")

    print()
    print("  Current indicator readings:")
    latest_row = model_df[oos_features].iloc[-1]
    for feat in oos_features:
        if feat in latest_row.index:
            print(f"    {feat:<16s} = {latest_row[feat]:>8.2f}")

    # Also show other model readings
    print()
    print("  All model readings:")
    for name, res_dict in results.items():
        p = res_dict["fitted"].iloc[-1] * 100
        print(f"    {name:<28s}: {p:.1f}%")

## Bonus: Estrella-Mishkin Quick Estimate

Using pre-estimated parameters from Estrella & Trubin (2006) for a
quick-and-dirty recession probability from the yield curve spread alone.
This requires no model fitting — just plug in the current spread.

In [ ]:
# ============================================================
# Estrella-Mishkin closed-form estimate
# Parameters from Estrella & Trubin (2006)
# P(recession) = Phi(-0.6045 - 0.7374 * spread)
# ============================================================
if "SPREAD" in data.columns:
    spread_latest = data["SPREAD"].dropna().iloc[-1]
    em_prob = stats.norm.cdf(-0.6045 - 0.7374 * spread_latest) * 100

    print(f"Current 10Y-3M spread:  {spread_latest:.2f}%")
    print(f"Estrella-Mishkin prob:  {em_prob:.1f}%")
    print()

    # Plot the Estrella-Mishkin curve: spread vs. probability
    spread_range = np.linspace(-4, 5, 200)
    em_curve = stats.norm.cdf(-0.6045 - 0.7374 * spread_range) * 100

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(spread_range, em_curve, "b-", linewidth=2)
    ax.axvline(x=spread_latest, color="red", linestyle="--", alpha=0.6,
               label=f"Current spread: {spread_latest:.2f}%")
    ax.axvline(x=0, color="gray", linestyle=":", alpha=0.4, label="Inversion point")
    ax.scatter([spread_latest], [em_prob], color="red", s=80, zorder=5)
    ax.annotate(f"{em_prob:.1f}%", (spread_latest, em_prob),
                textcoords="offset points", xytext=(15, 10), fontsize=11,
                arrowprops=dict(arrowstyle="->", color="red"))
    ax.set_xlabel("10Y-3M Treasury Spread (%)", fontsize=12)
    ax.set_ylabel("Recession Probability (%)", fontsize=12)
    ax.set_title("Estrella-Mishkin Yield Curve Model", fontsize=14, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.15)
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.show()

## Feature Importance Analysis

Which indicators contribute most to the model's recession predictions?
Measured by z-statistic magnitude (significance) and marginal effects.

In [ ]:
# ============================================================
# Feature importance: z-statistics and marginal effects
# ============================================================
if "Full Multi-Category" in results:
    res = results["Full Multi-Category"]["model"]

    # Extract coefficients and z-stats
    coef_df = pd.DataFrame({
        "Coefficient": res.params,
        "Std Error": res.bse,
        "z-stat": res.tvalues,
        "p-value": res.pvalues,
        "|z-stat|": res.tvalues.abs(),
    }).drop("const", errors="ignore")

    coef_df = coef_df.sort_values("|z-stat|", ascending=True)

    # Horizontal bar chart of z-statistics
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Panel 1: z-statistics
    colors = ["#d62728" if p < 0.05 else "#aec7e8" for p in coef_df["p-value"]]
    axes[0].barh(coef_df.index, coef_df["|z-stat|"], color=colors)
    axes[0].axvline(x=1.96, color="gray", linestyle="--", alpha=0.5,
                     label="5% significance")
    axes[0].set_xlabel("|z-statistic|")
    axes[0].set_title("Statistical Significance of Predictors")
    axes[0].legend()

    # Panel 2: Marginal effects at the mean
    mfx = res.get_margeff(at="mean")
    mfx_df = pd.DataFrame({
        "Marginal Effect": mfx.margeff,
        "Feature": [f for f in coef_df.index],
    }).set_index("Feature").sort_values("Marginal Effect")

    mfx_colors = ["#d62728" if v > 0 else "#2ca02c" for v in mfx_df["Marginal Effect"]]
    axes[1].barh(mfx_df.index, mfx_df["Marginal Effect"], color=mfx_colors)
    axes[1].axvline(x=0, color="gray", linewidth=0.8)
    axes[1].set_xlabel("Marginal Effect on P(Recession)")
    axes[1].set_title("Marginal Effects at the Mean")

    plt.suptitle("Feature Importance — Full Multi-Category Probit",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
    plt.show()

## Indicator Correlation Matrix

In [ ]:
# ============================================================
# Correlation heatmap of model features
# ============================================================
corr_cols = [f for f in FEATURES_FULL if f in model_df.columns]
corr_matrix = model_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)

ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(corr_cols, fontsize=9)

# Annotate cells
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}",
                ha="center", va="center", fontsize=7,
                color="white" if abs(corr_matrix.iloc[i, j]) > 0.6 else "black")

plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Historical Recession Detection Performance

How well did the model detect each historical recession 12 months in advance?

In [ ]:
# ============================================================
# Historical recession detection table
# ============================================================
if "Full Multi-Category" in results:
    fitted = results["Full Multi-Category"]["fitted"]

    # NBER recession peaks and troughs
    # Find recession start dates (transitions from 0 to 1 in USREC)
    usrec_model = model_df["USREC"]
    rec_starts = usrec_model[(usrec_model == 1) & (usrec_model.shift(1) == 0)].index
    rec_ends = usrec_model[(usrec_model == 0) & (usrec_model.shift(1) == 1)].index

    print(f"{"Recession":<25s} {"Peak Prob 12m Prior":<22s} {"Detected >30%?":<16s} {"Detected >50%?":<16s}")
    print("-" * 79)

    for start in rec_starts:
        # Look at model probability 6-18 months before recession start
        window_start = start - pd.DateOffset(months=18)
        window_end = start - pd.DateOffset(months=6)
        window_probs = fitted.loc[window_start:window_end]

        if len(window_probs) > 0:
            peak = window_probs.max() * 100
            det30 = "YES" if peak > 30 else "no"
            det50 = "YES" if peak > 50 else "no"
            print(f"  {start.strftime('%Y-%m'):<23s} {peak:<22.1f} {det30:<16s} {det50:<16s}")
        else:
            print(f"  {start.strftime('%Y-%m'):<23s} {"(no data)":<22s}")

## Export Results

Save probabilities and indicator data to CSV for further analysis.

In [ ]:
# ============================================================
# Export to CSV
# ============================================================
export_df = pd.DataFrame(index=model_df.index)
export_df["USREC"] = model_df["USREC"]
export_df["REC_12M_Actual"] = model_df[TARGET]

for name, res_dict in results.items():
    col_name = name.replace(" ", "_").replace("(", "").replace(")", "")
    export_df[f"Prob_{col_name}"] = res_dict["fitted"] * 100

export_df["Prob_OOS_Full"] = oos_probs * 100

# Add key indicators
for feat in FEATURES_FULL:
    if feat in model_df.columns:
        export_df[feat] = model_df[feat]

export_df.to_csv("recession_probabilities.csv")
print(f"Exported {len(export_df)} rows to recession_probabilities.csv")
export_df.tail()

## References

1. **Estrella, A. & Mishkin, F.S. (1998)**. "Predicting U.S. Recessions: Financial Variables as Leading Indicators." *Review of Economics and Statistics*, 80(1), 45-61.
2. **Wright, J.H. (2006)**. "The Yield Curve and Predicting Recessions." *Federal Reserve Board FEDS Working Paper* No. 2006-07.
3. **Kauppi, H. & Saikkonen, P. (2008)**. "Predicting U.S. Recessions with Dynamic Binary Response Models." *Review of Economics and Statistics*, 90(4), 777-791.
4. **Berge, T.J. (2014)**. "Predicting Recessions with Leading Indicators: Model Averaging and Links to the Financial Crisis." *Federal Reserve Bank of Kansas City Working Paper*.
5. **Federal Reserve Board FEDS Notes (2018, 2019)**. Various notes on recession probability models.
6. **Sahm, C. (2019)**. "Direct Stimulus Payments to Individuals." *Brookings Institution*.
7. **McCracken, M.W. & Ng, S. (2016)**. "FRED-MD: A Monthly Database for Macroeconomic Research." *Journal of Business & Economic Statistics*, 34(4), 574-589.

---

*Model inspired by Columbia Threadneedle's recession probability estimator.*
*Data sourced from FRED (Federal Reserve Bank of St. Louis).*